In [2]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 16.5 MB/s eta 0:00:0000:0100:01


In [3]:
import time
import requests
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import execute_batch
from datetime import datetime

In [4]:
URL = "https://api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": "usd",
    "ids": "bitcoin,ethereum,solana",
    "order": "market_cap_desc",
    "per_page": 10,
    "page": 1,
    "sparkline": False
}

POLL_INTERVAL = 15  # detik

In [5]:
resp = requests.get(URL, params=params, timeout=30)
resp.raise_for_status()
df = pd.json_normalize(resp.json())

# === BASIC CLEANING ===
df = df[[
"id", "symbol", "name",
"current_price", "market_cap",
"total_volume", "price_change_percentage_24h"]]

df.columns = ["crypto_id", "symbol", "name",
"price_usd", "market_cap",
"volume_24h", "price_change_24h"]

In [6]:
df

,crypto_id,symbol,name,price_usd,market_cap,volume_24h,price_change_24h
0,bitcoin,btc,Bitcoin,68433.00,1369850107964,47657729780,-3.19866
1,ethereum,eth,Ethereum,2046.30,247080431388,17439115423,-5.50417
2,solana,sol,Solana,85.54,48950940171,3771674949,-6.59339


In [8]:
conn = psycopg2.connect(
    host="postgres",
    port=5432,
    database="stream_db",
    user="airflow",
    password="airflow"
)
cur = conn.cursor()

In [ ]:
while True:
    try:
        resp = requests.get(URL, params=params, timeout=30)
        resp.raise_for_status()
        df = pd.json_normalize(resp.json())

        # === BASIC CLEANING ===
        df = df[[
            "id", "symbol", "name",
            "current_price", "market_cap",
            "total_volume", "price_change_percentage_24h"
        ]]

        df.columns = [
            "crypto_id", "symbol", "name",
            "price_usd", "market_cap",
            "volume_24h", "price_change_24h"
        ]

        # === HANDLE NULL ===
        df.fillna({
            "market_cap": 0,
            "volume_24h": 0,
            "price_change_24h": 0
        }, inplace=True)

        # === FEATURE ENGINEERING ===
        df["return_24h"] = df["price_change_24h"] / 100
        df["volatility_proxy"] = np.log1p(df["volume_24h"])
        df["fetched_at"] = datetime.utcnow()

        # === ANOMALY FLAG (Z-SCORE) ===
        std = df["price_change_24h"].std()
        mean = df["price_change_24h"].mean()

        if std == 0:
            df["anomaly_flag"] = 0
        else:
            z = (df["price_change_24h"] - mean) / std
            df["anomaly_flag"] = np.where(abs(z) > 3, 1, 0)

        # =============================
        # LOAD DIMENSION (SCD TYPE 1)
        # =============================
        dim_sql = """
        INSERT INTO crypto.dim_crypto (crypto_id, symbol, name, updated_at)
        VALUES (%s, %s, %s, NOW())
        ON CONFLICT (crypto_id)
        DO UPDATE SET
            symbol = EXCLUDED.symbol,
            name = EXCLUDED.name,
            updated_at = NOW();
        """

        dim_data = df[["crypto_id", "symbol", "name"]].values.tolist()
        execute_batch(cur, dim_sql, dim_data, page_size=50)

        # =============================
        # LOAD FACT (APPEND ONLY)
        # =============================
        fact_sql = """
        INSERT INTO crypto.fact_crypto_price (
            crypto_id,
            price_usd,
            market_cap,
            volume_24h,
            price_change_24h,
            return_24h,
            volatility_proxy,
            anomaly_flag,
            fetched_at
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s);
        """

        fact_data = df[[
            "crypto_id",
            "price_usd",
            "market_cap",
            "volume_24h",
            "price_change_24h",
            "return_24h",
            "volatility_proxy",
            "anomaly_flag",
            "fetched_at"
        ]].values.tolist()

        execute_batch(cur, fact_sql, fact_data, page_size=100)
        conn.commit()

        print(f"[{df['fetched_at'].iloc[0]}] inserted {len(df)} rows")

        time.sleep(POLL_INTERVAL)

    except Exception as e:
        conn.rollback()
        print("ERROR:", e)
        time.sleep(10)

[2026-03-26 20:10:40.727976] inserted 3 rows
[2026-03-26 20:10:55.899610] inserted 3 rows
[2026-03-26 20:11:10.997230] inserted 3 rows
[2026-03-26 20:11:26.291447] inserted 3 rows
[2026-03-26 20:11:41.879160] inserted 3 rows
ERROR: 429 Client Error: Too Many Requests for url: https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&ids=bitcoin%2Cethereum%2Csolana&order=market_cap_desc&per_page=10&page=1&sparkline=False
ERROR: 429 Client Error: Too Many Requests for url: https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&ids=bitcoin%2Cethereum%2Csolana&order=market_cap_desc&per_page=10&page=1&sparkline=False
ERROR: 429 Client Error: Too Many Requests for url: https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&ids=bitcoin%2Cethereum%2Csolana&order=market_cap_desc&per_page=10&page=1&sparkline=False
ERROR: 429 Client Error: Too Many Requests for url: https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&ids=bitcoin%2Cethereum%2Csolana&order=market_cap